# 02 — Data Preprocessing

Project: E-Waste Profitability Prediction
Group: 2026-AI-14
Module: IT3091 — Machine Learning

Objectives:
- Preserve the existing training/test split.
- Audit and clean invalid or inconsistent values.
- Record preprocessing decisions.
- Define fold-local imputation, encoding and scaling pipelines.
- Export cleaned training and test datasets.

The target is total_profit_usd.
No preprocessing parameters will be learned from the test dataset.

Libraries import

In [ ]:
import pandas as pd
import numpy as np
import json

from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

pd.set_option("display.max_columns", None)

In [ ]:
uploaded = files.upload()

Saving train_raw.csv to train_raw.csv
Saving test_raw.csv to test_raw.csv


In [ ]:
required_files = ["train_raw.csv", "test_raw.csv"]

for filename in required_files:
    assert filename in uploaded, f"Missing file: {filename}"

train_raw = pd.read_csv("train_raw.csv")
test_raw = pd.read_csv("test_raw.csv")

print("Training shape:", train_raw.shape)
print("Test shape:", test_raw.shape)

display(train_raw.head())

Training shape: (16000, 13)
Test shape: (4000, 13)


,device_id,item_name,device_category,condition,weight_g,device_age_years,gold_yield_g,silver_yield_g,copper_yield_g,total_profit_usd,Acquisition_Cost_LKR,Hazardous_Flag,Casing_Material
0,DEV-05895,Nikon D3500,Digital Camera,Used-Fair,440.58,7,0.0304,1.1019,42.93,2.53,13242.66,Yes,Plastic
1,DEV-03729,iPhone 11,Smartphone,Used-Good,179.41,8,0.0261,0.2686,17.22,1.84,11511.64,Yes,Mixed Metal
2,DEV-08959,Samsung Galaxy Watch 4,Smartwatch,New,42.33,3,0.0096,0.0749,3.94,0.94,14879.31,Yes,Aluminum
3,DEV-07672,iPhone 13,Smartphone,Used-Fair,178.62,6,0.0255,0.3385,17.12,1.46,19217.96,No,Aluminum
4,DEV-06000,Intel Core i3-10100,Microprocessor,E-Waste / End-of-Life,59.42,4,0.4903,0.1835,12.50,17.35,1252.62,No,Mixed Metal


Split and schema consistency

In [ ]:
expected_columns = [
    "device_id",
    "item_name",
    "device_category",
    "condition",
    "weight_g",
    "device_age_years",
    "gold_yield_g",
    "silver_yield_g",
    "copper_yield_g",
    "total_profit_usd",
    "Acquisition_Cost_LKR",
    "Hazardous_Flag",
    "Casing_Material"
]

assert train_raw.columns.tolist() == expected_columns
assert test_raw.columns.tolist() == expected_columns

for name, dataset in [
    ("Training", train_raw),
    ("Test", test_raw)
]:
    assert dataset["device_id"].notna().all()
    assert dataset["device_id"].is_unique

    print(
        f"{name} duplicate rows:",
        dataset.duplicated().sum()
    )

assert set(train_raw["device_id"]).isdisjoint(
    set(test_raw["device_id"])
)

assert len(train_raw) == 16000
assert len(test_raw) == 4000

print("Schema and split checks passed.")

Training duplicate rows: 0
Test duplicate rows: 0
Schema and split checks passed.


Reusable cleaning function

In [ ]:
numeric_columns = [
    "weight_g",
    "device_age_years",
    "gold_yield_g",
    "silver_yield_g",
    "copper_yield_g",
    "total_profit_usd",
    "Acquisition_Cost_LKR"
]

text_columns = [
    "device_id",
    "item_name",
    "device_category",
    "condition",
    "Hazardous_Flag",
    "Casing_Material"
]

nonnegative_inputs = [
    column for column in numeric_columns
    if column != "total_profit_usd"
]

def clean_dataset(dataset):
    cleaned = dataset.copy()
    audit = []

    for column in text_columns:
        original = cleaned[column].astype("string")
        stripped = original.str.strip()

        spaces_changed = int(
            original.ne(stripped).fillna(False).sum()
        )

        blank_count = int(
            stripped.eq("").fillna(False).sum()
        )

        stripped = stripped.mask(stripped.eq(""), pd.NA)

        # Use np.nan for sklearn categorical imputers.
        cleaned[column] = (
            stripped.astype(object)
            .where(stripped.notna(), np.nan)
        )

        audit.append({
            "column": column,
            "action": "Strip whitespace; blank to missing",
            "flagged_count": spaces_changed + blank_count
        })

    for column in numeric_columns:
        original = cleaned[column]
        converted = pd.to_numeric(original, errors="coerce")

        conversion_failures = int(
            (original.notna() & converted.isna()).sum()
        )

        infinite_count = int(
            converted.isin([np.inf, -np.inf]).sum()
        )

        cleaned[column] = converted.replace(
            [np.inf, -np.inf],
            np.nan
        )

        audit.append({
            "column": column,
            "action": "Numeric conversion; infinity to missing",
            "flagged_count": conversion_failures + infinite_count
        })

    for column in nonnegative_inputs:
        invalid = cleaned[column] < 0

        audit.append({
            "column": column,
            "action": "Negative physical input to missing",
            "flagged_count": int(invalid.sum())
        })

        cleaned.loc[invalid, column] = np.nan

    zero_weight = cleaned["weight_g"] == 0

    audit.append({
        "column": "weight_g",
        "action": "Zero weight to missing",
        "flagged_count": int(zero_weight.sum())
    })

    cleaned.loc[zero_weight, "weight_g"] = np.nan

    return cleaned, pd.DataFrame(audit)

Training and test datasets clean

In [ ]:
train_clean, train_cleaning_audit = clean_dataset(
    train_raw
)

test_clean, test_cleaning_audit = clean_dataset(
    test_raw
)

# Missing labels cannot be filled using an input imputer.
for name, dataset in [
    ("Training", train_clean),
    ("Test", test_clean)
]:
    assert dataset["total_profit_usd"].notna().all(), (
        f"{name} has missing/invalid targets. "
        "Review these records before proceeding."
    )

    assert dataset["device_id"].notna().all()
    assert dataset["device_id"].is_unique

# Cleaning must preserve the existing records and split.
assert train_clean["device_id"].tolist() == (
    train_raw["device_id"].astype(str).str.strip().tolist()
)

assert test_clean["device_id"].tolist() == (
    test_raw["device_id"].astype(str).str.strip().tolist()
)

assert set(train_clean["device_id"]).isdisjoint(
    set(test_clean["device_id"])
)

print("Training cleaning audit:")
display(train_cleaning_audit)

print("Test cleaning audit:")
display(test_cleaning_audit)

Training cleaning audit:


,column,action,flagged_count
0,device_id,Strip whitespace; blank to missing,0
1,item_name,Strip whitespace; blank to missing,0
2,device_category,Strip whitespace; blank to missing,0
3,condition,Strip whitespace; blank to missing,0
4,Hazardous_Flag,Strip whitespace; blank to missing,0
5,Casing_Material,Strip whitespace; blank to missing,0
6,weight_g,Numeric conversion; infinity to missing,0
7,device_age_years,Numeric conversion; infinity to missing,0
8,gold_yield_g,Numeric conversion; infinity to missing,0
9,silver_yield_g,Numeric conversion; infinity to missing,0


Test cleaning audit:


,column,action,flagged_count
0,device_id,Strip whitespace; blank to missing,0
1,item_name,Strip whitespace; blank to missing,0
2,device_category,Strip whitespace; blank to missing,0
3,condition,Strip whitespace; blank to missing,0
4,Hazardous_Flag,Strip whitespace; blank to missing,0
5,Casing_Material,Strip whitespace; blank to missing,0
6,weight_g,Numeric conversion; infinity to missing,0
7,device_age_years,Numeric conversion; infinity to missing,0
8,gold_yield_g,Numeric conversion; infinity to missing,0
9,silver_yield_g,Numeric conversion; infinity to missing,0


After cleaning check missing valus

In [ ]:
missing_summary = pd.DataFrame({
    "train_missing_before": train_raw.isna().sum(),
    "train_missing_after": train_clean.isna().sum(),
    "test_missing_before": test_raw.isna().sum(),
    "test_missing_after": test_clean.isna().sum()
})

display(missing_summary)

missing_summary.to_csv(
    "preprocessing_missing_summary.csv"
)

,train_missing_before,train_missing_after,test_missing_before,test_missing_after
device_id,0,0,0,0
item_name,0,0,0,0
device_category,0,0,0,0
condition,0,0,0,0
weight_g,0,0,0,0
device_age_years,0,0,0,0
gold_yield_g,0,0,0,0
silver_yield_g,0,0,0,0
copper_yield_g,0,0,0,0
total_profit_usd,0,0,0,0


Check Category consistency

In [ ]:
categorical_features = [
    "device_category",
    "condition",
    "Hazardous_Flag",
    "Casing_Material"
]

category_audit_rows = []

for column in categorical_features:
    train_values = set(
        train_clean[column].dropna().unique()
    )

    test_values = set(
        test_clean[column].dropna().unique()
    )

    unseen_test_values = sorted(
        test_values - train_values
    )

    category_audit_rows.append({
        "column": column,
        "training_categories": sorted(train_values),
        "unseen_test_categories": unseen_test_values
    })

    print(f"\n{column}")
    print("Training categories:", sorted(train_values))
    print("Unseen test categories:", unseen_test_values)

category_audit = pd.DataFrame(category_audit_rows)

category_audit.to_csv(
    "category_consistency_audit.csv",
    index=False
)


device_category
Training categories: ['Desktop Motherboard', 'Digital Camera', 'Hard Disk Drive', 'Laptop', 'Microprocessor', 'RAM Module', 'Smartphone', 'Smartwatch', 'Tablet', 'Wi-Fi Router']
Unseen test categories: []

condition
Training categories: ['E-Waste / End-of-Life', 'New', 'Refurbished', 'Used-Fair', 'Used-Good']
Unseen test categories: []

Hazardous_Flag
Training categories: ['No', 'Yes']
Unseen test categories: []

Casing_Material
Training categories: ['Aluminum', 'Mixed Metal', 'Plastic']
Unseen test categories: []


Outlier decision

## Outlier handling decision

IQR flags from the Data Understanding notebook identify extreme
values, but do not establish that those records are errors.

Valid high-weight and high-profit records are retained because
they may be important to the business decision.

No target clipping or automatic IQR-based row removal is applied.

Negative physical inputs and zero weight are marked as missing.
They should also be reviewed with the data provider.

Records where total metal yield exceeds device weight require
verification of units and measurement definitions. They are not
automatically deleted or corrected.

In [ ]:
metal_columns = [
    "gold_yield_g",
    "silver_yield_g",
    "copper_yield_g"
]

total_metal_g = train_clean[metal_columns].sum(
    axis=1,
    min_count=len(metal_columns)
)

metal_weight_flag = (
    total_metal_g > train_clean["weight_g"]
)

source_review_records = train_clean.loc[
    metal_weight_flag
].copy()

source_review_records["review_reason"] = (
    "Total metal yield exceeds recorded device weight"
)

display(source_review_records.head(10))

print(
    "Training records needing source review:",
    len(source_review_records)
)

,device_id,item_name,device_category,condition,weight_g,device_age_years,gold_yield_g,silver_yield_g,copper_yield_g,total_profit_usd,Acquisition_Cost_LKR,Hazardous_Flag,Casing_Material,review_reason


Training records needing source review: 0


Model inputs and target

In [ ]:
target_column = "total_profit_usd"

numeric_features = [
    "weight_g",
    "device_age_years",
    "gold_yield_g",
    "silver_yield_g",
    "copper_yield_g",
    "Acquisition_Cost_LKR"
]

feature_columns = (
    numeric_features + categorical_features
)

excluded_columns = [
    "device_id",
    "item_name"
]

assert target_column not in feature_columns
assert not set(excluded_columns) & set(feature_columns)

X_train = train_clean[feature_columns].copy()
y_train = train_clean[target_column].copy()

X_test = test_clean[feature_columns].copy()
y_test = test_clean[target_column].copy()

print("Training inputs:", X_train.shape)
print("Test inputs:", X_test.shape)
print("Training target:", y_train.shape)

Training inputs: (16000, 10)
Test inputs: (4000, 10)
Training target: (16000,)


Leakage-safe preprocessing pipeline

In [ ]:
def make_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    return ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ])

preprocessor = make_preprocessor(
    numeric_features,
    categorical_features
)

print("Preprocessor defined successfully.")
print("It has NOT been fitted.")

Preprocessor defined successfully.
It has NOT been fitted.


Preprocessing Decision Log

In [ ]:
preprocessing_log = pd.DataFrame([
    {
        "issue": "Train/test split",
        "decision": "Preserve existing 80/20 split",
        "reason": "Fair comparison across all models"
    },
    {
        "issue": "Missing numerical inputs",
        "decision": "Median imputation inside model pipeline",
        "reason": "Fit only on each CV training fold"
    },
    {
        "issue": "Missing categories",
        "decision": "Most-frequent imputation inside pipeline",
        "reason": "Fold-local fitting prevents leakage"
    },
    {
        "issue": "Category encoding",
        "decision": "One-hot encoding; ignore unseen categories",
        "reason": "Avoid artificial ordering of nominal categories"
    },
    {
        "issue": "Scaling",
        "decision": "StandardScaler inside model pipeline",
        "reason": "Supports scale-sensitive models; fold-local fit"
    },
    {
        "issue": "Outliers",
        "decision": "Retain valid extreme values",
        "reason": "Extreme values are not automatically errors"
    },
    {
        "issue": "Invalid measurements",
        "decision": "Negative inputs and zero weight to missing",
        "reason": "Physically invalid values require review"
    },
    {
        "issue": "Missing target",
        "decision": "Stop and review; never impute target",
        "reason": "Inventing labels would invalidate evaluation"
    },
    {
        "issue": "Identifiers",
        "decision": "Exclude device_id and initially item_name",
        "reason": "Avoid identifier-driven memorisation"
    },
    {
        "issue": "Currencies",
        "decision": "Retain separate LKR and USD columns",
        "reason": "No unsupported currency subtraction"
    },
    {
        "issue": "Class imbalance",
        "decision": "Assess after priority labels are defined",
        "reason": "Use fold-local methods and suitable metrics later"
    }
])

display(preprocessing_log)

,issue,decision,reason
0,Train/test split,Preserve existing 80/20 split,Fair comparison across all models
1,Missing numerical inputs,Median imputation inside model pipeline,Fit only on each CV training fold
2,Missing categories,Most-frequent imputation inside pipeline,Fold-local fitting prevents leakage
3,Category encoding,One-hot encoding; ignore unseen categories,Avoid artificial ordering of nominal categories
4,Scaling,StandardScaler inside model pipeline,Supports scale-sensitive models; fold-local fit
5,Outliers,Retain valid extreme values,Extreme values are not automatically errors
6,Invalid measurements,Negative inputs and zero weight to missing,Physically invalid values require review
7,Missing target,Stop and review; never impute target,Inventing labels would invalidate evaluation
8,Identifiers,Exclude device_id and initially item_name,Avoid identifier-driven memorisation
9,Currencies,Retain separate LKR and USD columns,No unsupported currency subtraction


Cleaned datasets සහ logs save

In [ ]:
train_clean.to_csv(
    "train_clean.csv",
    index=False
)

test_clean.to_csv(
    "test_clean.csv",
    index=False
)

train_cleaning_audit.to_csv(
    "train_cleaning_audit.csv",
    index=False
)

test_cleaning_audit.to_csv(
    "test_cleaning_audit.csv",
    index=False
)

source_review_records.to_csv(
    "preprocessing_source_review.csv",
    index=False
)

preprocessing_log.to_csv(
    "preprocessing_decision_log.csv",
    index=False
)

preprocessing_config = {
    "target": target_column,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "excluded_columns": excluded_columns,
    "numeric_imputation": "median, fitted inside CV pipeline",
    "categorical_imputation": (
        "most_frequent, fitted inside CV pipeline"
    ),
    "encoding": "one-hot, handle_unknown=ignore",
    "scaling": "StandardScaler, fitted inside CV pipeline",
    "split_preserved": True,
    "training_rows": len(train_clean),
    "test_rows": len(test_clean)
}

with open("preprocessing_config.json", "w") as file:
    json.dump(
        preprocessing_config,
        file,
        indent=2
    )

print("Cleaned datasets and preprocessing evidence saved.")

Cleaned datasets and preprocessing evidence saved.


Evidence download

In [ ]:
files.download("preprocessing_decision_log.csv")
files.download("preprocessing_config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("train_clean.csv")
files.download("test_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>